# Transaction-log sweep on Colab

Replicates the Bank Transactions result from *Let's (not) just put things in Context*
(arXiv 2512.13898): in-context and thinking accuracy collapse as the log grows, while
query-only test-time training holds up.

**Run the gate first.** Cell 5 runs `txlog_gate.yaml` — about ten minutes. The paper's
result needs the shortest context well above the accuracy floor (Qwen3-4B scores 36% at
its shortest setting), and a model already near zero at 25 transactions cannot show a
crossover no matter how well qTTT works. Check that number before paying for cell 6.

**Disconnects are free.** `runs/` and `.cache/` live on Drive and the runner skips any
`(uid, arrangement, model)` row already in `records.jsonl`, so re-running a cell after a
disconnect picks up where it stopped.

## 1. What GPU did we get?

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"{gb:.0f} GB visible, bf16={torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")

# Peak usage is roughly: weights + AdamW states + KV cache + span-backward
# activations. For Qwen3-4B at ~19k tokens (500 transactions in this repo's line
# format) that is about 8 + 4.5 + 2.8 + 5.6 = 21 GB.
#
# That 8 GB is ONE copy. The runner builds every model in a config up front, so
# the three arms would otherwise hold three copies of the same checkpoint --
# 24 GB of weights before a single activation, which does not fit. Arms sharing
# a checkpoint share the loaded weights; without that this config OOMs an A100.
if gb >= 30:
    print("\n-> Qwen3-4B, windows 25/95/250/500. Run the configs as committed.")
elif gb >= 20:
    print("\n-> Qwen3-4B fits only with `optimizer: sgd` in the qttt arm's `extra`,")
    print("   and consider dropping tx_window_500. AdamW keeps two fp32 states per")
    print("   query projection: ~4.5 GB at 4B.")
else:
    print("\n-> Too small for 4B. Set model to Qwen/Qwen3-1.7B and use windows")
    print("   25/50/95. Trend replication only; not comparable to Table 2.")
print("\nThis cell only advises. Edit the YAML yourself so the run records what you chose.")

## 2. Persist `runs/` and `.cache/` on Drive

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
STORE = Path("/content/drive/MyDrive/ctxlab")
(STORE / "runs").mkdir(parents=True, exist_ok=True)
(STORE / "cache").mkdir(parents=True, exist_ok=True)
print("persisting to", STORE)

## 3. Install

In [ ]:
# pip, not `uv sync --extra local`: uv would build a fresh venv with a torch that
# does not match Colab's CUDA driver. The uv flow stays for local dev and CI.
BRANCH = "arr/qttt"
FORK = "https://github.com/zach-yaninek/sundai_TTT.git"

!git clone --branch {BRANCH} --depth 1 {FORK} /content/ctxlab 2>&1 | tail -2
%cd /content/ctxlab
!pip install -q -e . 2>&1 | tail -2
!python -c "import ctxlab, transformers, torch; print('ctxlab', ctxlab.__version__, '| torch', torch.__version__)"

In [ ]:
from pathlib import Path
import os

# Symlink so a disconnect costs nothing: the runner resumes from records.jsonl.
for name, target in (("runs", STORE / "runs"), (".cache", STORE / "cache")):
    link = Path("/content/ctxlab") / name
    if not link.is_symlink():
        if link.exists():
            import shutil; shutil.rmtree(link)
        link.symlink_to(target)
print("runs ->", Path("runs").resolve())

# Qwen3 is ungated, so this is optional.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

## 5. Gate — about ten minutes

Land in the 20–40% band on `tx_joint` and the sweep is worth running. Near zero means
reach for a larger model first, before spending hours on qTTT.

In [ ]:
!ctxlab run -c configs/experiments/txlog_gate.yaml

## 6. The sweep

Roughly 5–6 GPU-hours at `n: 100` on an A100. Start smaller by editing `dataset.n` if you
want a first signal sooner. Safe to re-run after a disconnect.

In [ ]:
!ctxlab run -c configs/experiments/txlog_replication.yaml

## 7. Results

In [ ]:
!ctxlab report runs/txlog_replication

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

rows = [json.loads(l) for l in open("runs/txlog_replication/records.jsonl")]
acc, toks = defaultdict(list), defaultdict(list)
for r in rows:
    acc[(r["arrangement"], r["model"])].append(r["metrics"]["tx_joint"])
    if r.get("usage", {}).get("input_tokens"):
        toks[r["arrangement"]].append(r["usage"]["input_tokens"])

windows = sorted({a for a, _ in acc}, key=lambda a: int(a.split("_")[-1]))
arms = sorted({m for _, m in acc})
head = f"{'window':>14} {'tokens':>8} " + " ".join(f"{m:>22}" for m in arms)
print(head); print("-" * len(head))
for w in windows:
    t = sum(toks[w]) / len(toks[w]) if toks[w] else 0
    cells = []
    for m in arms:
        v = acc.get((w, m))
        cells.append(f"{100*sum(v)/len(v):>21.1f}%" if v else f"{'-':>22}")
    print(f"{w:>14} {t:>8.0f} " + " ".join(cells))

print("\nThe claim is the shape: in-context and thinking should fall steeply with")
print("window size while qTTT falls slowly, crossing over somewhere after the")
print("shortest window. Absolute values differ from the paper's Table 2 -- see the")
print("token-density note in docs/research-log.md.")